In [1]:
import sys
sys.path.append('../')

%env MUJOCO_GL=egl

env: MUJOCO_GL=egl


In [2]:
import os
import jax
import jax.numpy as jnp
import mujoco
import numpy as np
import mediapy as media
from dataclasses import dataclass, field
from mujoco.mjx._src import math as mjx_math

from builderbench.env_utils import make_env
from utils.wrapper import wrap_env
from utils.networks import load_params

In [3]:
AGENT = "ppdpo"

In [4]:
from ppdpo import Args as ALGArgs
@dataclass
class Args(ALGArgs):
    env_id: str = 'creative-4-task1'
    
    folder_path: str = "checkpoints/"
    fps: int = 10
    num_envs: int = 1

In [5]:
args = Args()

In [6]:
from utils.wrapper import Wrapper

@jax.jit
def get_yaw_from_quat(q):
    w, x, y, z = q[0], q[1], q[2], q[3]
    siny_cosp = 2 * (w * z + x * y)
    cosy_cosp = 1 - 2 * (y * y + z * z)
    yaw = jnp.arctan2(siny_cosp, cosy_cosp)
    return yaw
    
class PDWrapper(Wrapper):
    def __init__(self, env, duration: int = 1, kp_pos: float = 10.0, kd_pos: float = 2.0, kp_yaw: float = 10.0, kd_yaw: float = 0.5, delta_control: bool = False):
        super().__init__(env)
        
        self._kp_pos = kp_pos
        self._kd_pos = kd_pos
        self._kp_yaw = kp_yaw
        self._kd_yaw = kd_yaw
        self._duration = duration
        self._delta_control = delta_control
        
        # to do, port the following values from env config
        self._cube_mass = 0.07936
        self._gravity = 9.81
        self._gravity_comp = self._cube_mass * self._gravity

        self._workspace_median = (self.env._workspace_bounds[1] + self.env._workspace_bounds[0]) / 2
        self._workspace_halfspan = (self.env._workspace_bounds[1] - self.env._workspace_bounds[0]) / 2

        self._yaw_median = jnp.array( [0.0] )
        self._yaw_halfspan = jnp.array( [np.pi / 2] )

    def get_action(self, state, waypoint_pos, waypoint_yaw, cube_id):
        current_pos = state.data.qpos[self.env._objs_qposadr[:, None] + np.arange(3)][cube_id]
        current_quat = state.data.qpos[(self.env._objs_qposadr + 3)[:, None] + np.arange(4)][cube_id]
        current_linvel = state.data.qvel[self.env._objs_qveladr[:, None] + np.arange(3)][cube_id]
        current_angvel = state.data.qvel[(self.env._objs_qveladr + 3)[:, None] + np.arange(3)][cube_id]
        
        error_pos = waypoint_pos - current_pos
        output_pos = (self._kp_pos * error_pos) + (self._kd_pos * - current_linvel)
        output_pos = output_pos.at[2].add(self._gravity_comp)

        current_yaw = get_yaw_from_quat(current_quat)
        delta_yaw = waypoint_yaw - current_yaw
        error_yaw = jnp.arctan2(jnp.sin(delta_yaw), jnp.cos(delta_yaw))        
        output_yaw = (self._kp_yaw * error_yaw) + (self._kd_yaw * - current_angvel[-1])

        raw_ctrl_action = jnp.concatenate([output_pos, output_yaw], axis=0)
        ctrl_action = ( raw_ctrl_action - self.env._ctrl_median[:4] ) / self.env._ctrl_halfspan[:4]
        
        select_action = ( ( ( 2 * cube_id + 1) * jnp.pi / self.env._config.num_cubes ) - jnp.pi ) / ( jnp.pi )

        action =  jnp.concatenate([ctrl_action, select_action[None]], axis=0)
        action = jnp.clip(action, -1, 1)
        return action
    
    def step(self, state, action):

        state.info.update(
            select_action = jnp.clip(action[-1], -1, 1),
        )
        cube_id = jnp.digitize( ( self.env._action_scale[-1] * action[-1] + jnp.pi ), bins = jnp.arange(1, self.env._config.num_cubes+1) * 2 * jnp.pi / ( self.env._config.num_cubes ) )

        if self._delta_control:
            raise NotImplementedError
        else:
            waypoint_pos = action[:3] * self._workspace_halfspan + self._workspace_median
            waypoint_yaw = action[3] * self._yaw_halfspan + self._yaw_median

        def f(carry, _):
            state, prev_done = carry
            action = self.get_action(state, waypoint_pos, waypoint_yaw, cube_id)
            state = self.env.step(state, action)
            done = jnp.maximum(state.done, prev_done)
            
            return (state, done), (state.reward, prev_done, state.metrics)
        
        (state, final_done), (rewards, dones, metrics) = jax.lax.scan(f, (state, state.done), (), self._duration)

        state = state.replace(reward = jnp.sum(rewards * (1-dones)))
        state = state.replace(metrics = jax.tree_util.tree_map(lambda m: jnp.sum( m * (1-dones) ), metrics))
        state = state.replace(done = final_done)
        return state    

    def step_with_info(self, state, action):

        state.info.update(
            select_action = jnp.clip(action[-1], -1, 1),
        )
        cube_id = jnp.digitize( ( self.env._action_scale[-1] * action[-1] + jnp.pi ), bins = jnp.arange(1, self.env._config.num_cubes+1) * 2 * jnp.pi / ( self.env._config.num_cubes ) )

        if self._delta_control:
            raise NotImplementedError
        else:
            waypoint_pos = action[:3] * self._workspace_halfspan + self._workspace_median
            waypoint_yaw = action[3] * self._yaw_halfspan + self._yaw_median

        def f(carry, _):
            state, prev_done = carry
            action = self.get_action(state, waypoint_pos, waypoint_yaw, cube_id)
            state = self.env.step(state, action)
            done = jnp.maximum(state.done, prev_done)
            
            return (state, done), (state.reward, prev_done, state.metrics, state)
        
        (state, final_done), (rewards, dones, metrics, states) = jax.lax.scan(f, (state, state.done), (), self._duration)

        state = state.replace(reward = jnp.sum(rewards * (1-dones)))
        state = state.replace(metrics = jax.tree_util.tree_map(lambda m: jnp.sum( m * (1-dones) ), metrics))
        state = state.replace(done = final_done)
        return state, states    

In [7]:
np.random.seed(args.seed)
key = jax.random.PRNGKey(args.seed)
key, key_env, key_eval, key_policy, key_value = jax.random.split(key, 5)

In [8]:
env_class, default_config = make_env(args)
env = env_class(config=default_config)
env = PDWrapper(env, duration=args.duration)
new_episode_length = default_config.episode_length // args.duration

Warp 1.9.0 initialized:
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:1"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:2"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:3"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:4"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:5"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:6"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:7"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
   CUDA peer access:
     Supported fully (all-directional)
   Kernel cache:
     /home/nvidia/.cache/warp/1.9.0


In [9]:
action_size = env.action_size

from utils.evaluation import get_video, Evaluator

from ppdpo import PPONetworks, Actor, Value
ppo_network = PPONetworks( 
    policy_network = Actor(layer_sizes=args.policy_hidden_sizes + [action_size * 2]),
value_network = Value(layer_sizes=args.value_hidden_sizes  + [1]),
)

from ppdpo import make_inference_fn
make_policy = make_inference_fn(ppo_network)

run_name = "discount-0.9__creative-4-task1__43__ppdpo__1764119874"
path_name = "discount-0.9__creative-4-task1__43__ppdpo__1764119874/params_40"
params = load_params(f"../checkpoints/{path_name}.pkl")
actor_params, _, normalize_params = params

jit_inference_fn = jax.jit(
                    make_policy(
                        {
                            'policy': actor_params, 
                            'normalizer': normalize_params,
                        },
                        deterministic=True,
                    )
                )

In [10]:
reset_fn = jax.jit(env.reset)
step_fn = jax.jit(env.step_with_info)

In [11]:
rollout = []
returns = []
env_state = reset_fn(key_env)
rollout.append(env_state)

for i in range(new_episode_length):

    key_policy, key = jax.random.split(key)
    action, _ = jit_inference_fn(env_state.obs, env_state.info["target_goal"], key_policy) 
    
    # waypoint_pos = action[:3] * env._workspace_halfspan + env._workspace_median
    # waypoint_yaw = action[3] * env._yaw_halfspan + env._yaw_median
    # action = action.at[:3].set( ( env_state.info["target_goal"] - env._workspace_median ) / env._workspace_halfspan )
    # print(action)
    
    env_state, state_info = step_fn(env_state, action)
    rollout.append(state_info)

    returns.append( env_state.reward )

Module mujoco.mjx.third_party.mujoco_warp._src.smooth f8aea15 load on device 'cuda:0' took 18.81 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver abf11d9 load on device 'cuda:0' took 1.46 ms  (cached)
Module _nxn_broadphase__locals__kernel_693f1c65 693f1c6 load on device 'cuda:0' took 0.98 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_primitive._create_narrowphase_kernel 64aeafa load on device 'cuda:0' took 2.31 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 314db63 load on device 'cuda:0' took 4.34 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward ea0168c load on device 'cuda:0' took 3.08 ms  (cached)
Module _create_actuator_velocity_kernel__locals__actuator_velocity_4b9adcc9 9173f9a load on device 'cuda:0' took 2.37 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.passive c66cecd load on device 'cuda:0' took 1.39 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.support 

In [12]:
camera = mujoco.MjvCamera()
camera.distance = 0.8
camera.lookat = np.array([0.4, 0.0 , 0.4])
camera.elevation = -30.0
camera.azimuth = 180

In [13]:
video_images = []
mocap_key = 'target_mocap'
for i in range(new_episode_length):
    
    roll_ = rollout[i]

    if i == 0:
        video_images.append(
            env.render_from_info(
                rollout[i].data.qpos,
                rollout[i].data.qvel, 
                rollout[i].info[f'{mocap_key}_pos'],
                rollout[i].info[f'{mocap_key}_quat'],
                camera=camera,
            )
        )
        
    else:

        for d in range( args.duration ):
            t = args.duration*i + d
        
            if t % 2 == 0:
                video_images.append(
                    env.render_from_info(
                        roll_.data.qpos[d],
                        roll_.data.qvel[d], 
                        roll_.info[f'{mocap_key}_pos'][d],
                        roll_.info[f'{mocap_key}_quat'][d],
                        camera=camera,
                    )
                )

In [14]:
media.show_video(video_images, fps=1.0 / env.dt / 2)

This browser does not support the video tag.


In [15]:
os.makedirs(f"../videos/{run_name}", exist_ok=True)

media.write_video(
    f"../videos/{path_name}.mp4",
    video_images,
    fps=1.0 / env.dt / 2
)

In [41]:
media.show_video(video_images, fps=1.0 / env.dt / 2)

This browser does not support the video tag.
